# Sort and split the large trihydroxy MGFs before Falcon

The C24 alkamine trihydroxy MGF (~1.5 M spectra) is sorted by precursor m/z, renumbered and split into parts so Falcon can cluster it; each part is renumbered again. (`split_mgf.py` is the command-line version of the same steps.)

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

## Reorder trihydorxy mgf based on pepmass

In [ ]:
# sort_mgf_by_pepmass.py
# Usage:  python sort_mgf_by_pepmass.py input.mgf output.mgf

import sys
from tqdm import tqdm

def sort_mgf_by_massql_pepmass(infile, outfile):
    # First, read the entire file
    with open(infile, "r") as f:
        lines = f.readlines()

    # Split into blocks
    blocks_raw = []
    block = []
    for line in lines:
        if line.startswith("BEGIN IONS"):
            block = [line]
        elif line.startswith("END IONS"):
            block.append(line)
            blocks_raw.append(block)
        else:
            block.append(line)

    # Parse blocks with progress bar
    blocks = []
    for block in tqdm(blocks_raw, desc="Parsing and sorting"):
        massql_pepmass = None
        for line in block:
            if line.startswith("MASSQL_PEPMASS="):
                massql_pepmass = float(line.split("=", 1)[1].strip())
                break
        if massql_pepmass is None:
            raise ValueError("MASSQL_PEPMASS missing in a block")
        blocks.append((massql_pepmass, "".join(block)))

    # Sort and write
    blocks.sort(key=lambda x: x[0])
    with open(outfile, "w") as fout:
        for _, block in blocks:
            fout.write(block)

In [ ]:
input_mgf = f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge.mgf"
output_mgf = f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass.mgf"

sort_mgf_by_massql_pepmass(input_mgf, output_mgf)

## Renumber scans

In [ ]:
#Then renumber the scans in this new mgf

in_file = f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass.mgf'

with open(in_file, 'r') as in_f:
    with open(f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_renumbered.mgf',
              'a') as out_f:
        counter = 1
        for line in in_f:
            if line.startswith('SCANS'):
                print('SCANS=' + str(counter), file=out_f, end='\n')
                counter += 1
            else:
                print(line, file=out_f, end='')

## Split big trihydroxy mgf into three equal parts (if possible) and renumber

In [ ]:
def split_mgf_by_equal_parts(input_mgf, output1, output2, output3):
    with open(input_mgf, "r") as f:
        lines = f.readlines()

    # First pass: collect all blocks and count total scans
    blocks = []
    current_block = []
    
    for line in lines:
        if line.startswith("BEGIN IONS"):
            current_block = [line]
        elif line.startswith("END IONS"):
            current_block.append(line)
            blocks.append("".join(current_block))
            current_block = []
        else:
            current_block.append(line)
    
    total_scans = len(blocks)
    print(f"Total scans found: {total_scans}")
    
    # Calculate split points for 3 equal parts
    part1_size = total_scans // 3
    part2_size = total_scans // 3
    part3_size = total_scans - part1_size - part2_size  # Remainder goes to part3
    
    print(f"Part 1: {part1_size} scans")
    print(f"Part 2: {part2_size} scans")
    print(f"Part 3: {part3_size} scans")
    
    # Split blocks
    blocks_1 = blocks[:part1_size]
    blocks_2 = blocks[part1_size:part1_size + part2_size]
    blocks_3 = blocks[part1_size + part2_size:]
    
    # Helper function to renumber scans in a block
    def renumber_scan_in_block(block_text, new_scan_number):
        lines = block_text.split('\n')
        renumbered_lines = []
        
        for line in lines:
            if line.startswith("SCANS="):
                renumbered_lines.append(f"SCANS={new_scan_number}")
            else:
                renumbered_lines.append(line)
        
        return '\n'.join(renumbered_lines)
    
    # Write outputs with renumbered scans
    with open(output1, "w") as f1:
        for i, block in enumerate(blocks_1, 1):
            renumbered_block = renumber_scan_in_block(block, i)
            f1.write(renumbered_block)
    
    with open(output2, "w") as f2:
        for i, block in enumerate(blocks_2, 1):
            renumbered_block = renumber_scan_in_block(block, i)
            f2.write(renumbered_block)
    
    with open(output3, "w") as f3:
        for i, block in enumerate(blocks_3, 1):
            renumbered_block = renumber_scan_in_block(block, i)
            f3.write(renumbered_block)
    
    print(f"Successfully split MGF file into 3 parts:")
    print(f"  {output1}: {len(blocks_1)} scans")
    print(f"  {output2}: {len(blocks_2)} scans")
    print(f"  {output3}: {len(blocks_3)} scans")


# Usage
split_mgf_by_equal_parts(
    input_mgf=f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_renumbered.mgf",
    output1=f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part1_test.mgf",
    output2=f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part2_test.mgf",
    output3=f"{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part3_test.mgf"
)

## Finally, renumber again

In [ ]:
#Then renumber the scans in these new mgfs, individually

in_file = f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part1_test.mgf'

with open(in_file, 'r') as in_f:
    with open(f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part1_test_renumbered.mgf',
              'a') as out_f:
        counter = 1
        for line in in_f:
            if line.startswith('SCANS'):
                print('SCANS=' + str(counter), file=out_f, end='\n')
                counter += 1
            else:
                print(line, file=out_f, end='')

print("total number of scans: ", counter - 1)

In [ ]:
#Then renumber the scans in these new mgfs, individually

in_file = f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part2_test.mgf'

with open(in_file, 'r') as in_f:
    with open(f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part2_test_renumbered.mgf',
              'a') as out_f:
        counter = 1
        for line in in_f:
            if line.startswith('SCANS'):
                print('SCANS=' + str(counter), file=out_f, end='\n')
                counter += 1
            else:
                print(line, file=out_f, end='')

print("total number of scans: ", counter - 1)

In [ ]:
#Then renumber the scans in these new mgfs, individually

in_file = f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part3_test.mgf'

with open(in_file, 'r') as in_f:
    with open(f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_sorted_pepmass_part3_test_renumbered.mgf',
              'a') as out_f:
        counter = 1
        for line in in_f:
            if line.startswith('SCANS'):
                print('SCANS=' + str(counter), file=out_f, end='\n')
                counter += 1
            else:
                print(line, file=out_f, end='')

print("total number of scans: ", counter - 1)